# پروژهٔ دستیار پیگیری پرونده

راهنما: این نوت‌بوک مسیر داده، بازیابی، پاسخ مستند، تأیید انسانی، آزمون و ارزیابی را مانند تمرین‌ها اجرا می‌کند. حالت پیش‌فرض `replay` است و هیچ کلید یا هزینه‌ای ندارد. خروجی آزمایشی، نتیجهٔ مدل واقعی محسوب نمی‌شود.

In [1]:
from pathlib import Path
import sys, json, subprocess, tempfile
ROOT = Path.cwd()
if not (ROOT / 'src' / 'casepilot').exists():
    raise RuntimeError('نوت‌بوک را از ریشهٔ پروژه باز کنید.')
sys.path.insert(0, str(ROOT / 'src'))
from casepilot.common import read_json
from casepilot.store import Store
from casepilot.retrieval import Retriever
from casepilot.agent import Agent
from casepilot.model import ReplayClient
print('وضعیت: محیط آماده است.')

وضعیت: محیط آماده است.


## مرحلهٔ داده و مرز ارزیابی

توضیح: گزارش‌های واقعی از دادهٔ ثابت خوانده می‌شوند. پرونده‌های ارزیابی، نظرها و خانواده‌های تکراری‌شان پیش از قطعه‌بندی از بازیابی کنار گذاشته شده‌اند. ارزیابی روی snapshot کنونی است و ادعای بازسازی تاریخی ندارد.

In [2]:
manifest = read_json(ROOT / 'data' / 'snapshot_manifest.json')
print(json.dumps({k: manifest[k] for k in ('issue_count','comments_retained','docs_count','chunks','split','excluded_docs')}, ensure_ascii=False, indent=2))
cases = read_json(ROOT / 'eval' / 'cases.json')
print('تعداد پرونده‌های توسعه:', sum(c['split']=='dev' for c in cases))

{
  "issue_count": 245,
  "comments_retained": 203,
  "docs_count": 39,
  "chunks": 1997,
  "split": {
    "dev": 15,
    "test": 15
  },
  "excluded_docs": []
}
تعداد پرونده‌های توسعه: 15


## مرحلهٔ بازیابی و استناد

توضیح: روش پایه جست‌وجوی واژگانی است. روش نهایی ادغام رتبهٔ واژه و سه‌نویسه، گسترش واژگان و تنوع منبع را اضافه می‌کند. کنترل خودکار، وجود نقل‌قول در منبع را می‌سنجد؛ تشخیص کامل صحت معنایی همچنان نیازمند بازبینی است.

In [3]:
retriever = Retriever()
query = 'session_state widget value resets after navigation'
for method in ('baseline','final'):
    rows = retriever.search(query, method=method)
    print('روش:', method)
    print([(r['source_id'], r['section']) for r in rows])

روش: baseline
[('issue:11266', 'Summary'), ('issue:11266', 'Current Behavior'), ('issue:11266', 'Expected Behavior'), ('issue:7206', 'Checklist'), ('docs:widget-behavior', "Changing a widget's identity will reset it")]
روش: final
[('docs:session_state', 'Session State and Widget State association'), ('docs:widget-updating-session-state', 'Solution'), ('docs:widget-behavior', "Retain statefulness when changing a widget's identity"), ('docs:widgets', 'Option 2: Save your widget values into a dummy key in Session State'), ('docs:session-state', 'Session State and Widget State association')]


## مرحلهٔ پاسخ و گفت‌وگوی وابسته

توضیح: اطلاعات تازه جای واقعیت قبلی را می‌گیرد و در سابقه ثبت می‌شود. هیچ نظر عمومی در این مرحله ثبت نمی‌شود.

In [4]:
temporary = tempfile.TemporaryDirectory(prefix='casepilot-notebook-')
store = Store(Path(temporary.name) / 'tracker.sqlite3')
agent = Agent(store, retriever, ReplayClient())
first = agent.turn('notebook-case', 'مقدار session_state پس از جابه‌جایی صفحه تغییر می‌کند.', 'nb-turn-1')
print(first['response'])
second = agent.turn('notebook-case', 'پاسخ: نسخه مشخص شد و نمونهٔ کوچک هم رفتار را دارد.', 'nb-turn-2', facts={'streamlit_version':'1.49.0','reproducible':True})
print(second['response'])
assert len(store.get('notebook-case')['comments']) == 0

پاسخ پیشنهادی: پرسش بعدی: نسخهٔ دقیق `Streamlit` در محیطی که خطا رخ می‌دهد چیست؟

شاهد مستند: متن رسمی؛ سازگاری نسخه باید بررسی شود.

بخش منبع:

```text
Session State and Widget State association
```

نسخهٔ منبع:

```text
695e6ce2fb1b23afc6df66b40420ab0013ad72fd
```

متن شاهد:

```text
Every widget with a key is automatically added to Session State:
```

نشانی منبع:

```text
https://github.com/streamlit/docs/blob/695e6ce2fb1b23afc6df66b40420ab0013ad72fd/content/develop/api-reference/caching-and-state/session_state.md#L84-L90
```

شاهد مستند: متن رسمی؛ سازگاری نسخه باید بررسی شود.

بخش منبع:

```text
Navigation
```

نسخهٔ منبع:

```text
695e6ce2fb1b23afc6df66b40420ab0013ad72fd
```

متن شاهد:

```text
**Secondary recommendation:** For widgets that must be on individual pages, use the placeholder key pattern.
```

نشانی منبع:

```text
https://github.com/streamlit/docs/blob/695e6ce2fb1b23afc6df66b40420ab0013ad72fd/content/develop/concepts/architecture/widget-behavior.md#L324-L333
```

محدو

## مرحلهٔ تأیید انسانی و اقدام

راهنما: متن پیشنهاد را بخوانید. مقدار `REVIEW_DECISION` را خودتان انتخاب کنید. مقدار پیش‌فرض `reject` است. تأیید همین متن و همین پرونده معتبر است و ویرایش نیازمند تأیید تازه است.

In [5]:
REVIEW_DECISION = 'reject'
p = second['proposal']
review = store.review('notebook-case', p['id'], p['hash'], REVIEW_DECISION, 'بازبین نوت‌بوک')
if REVIEW_DECISION == 'approve':
    result = store.execute('notebook-case', p['id'], review['approval_id'])
    repeated = store.execute('notebook-case', p['id'], review['approval_id'])
    assert repeated['replayed']
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    assert len(store.get('notebook-case')['comments']) == 0
    print('وضعیت: پیشنهاد رد شد و تغییری ثبت نشد.')

وضعیت: پیشنهاد رد شد و تغییری ثبت نشد.


## مرحلهٔ آزمون و ارزیابی توسعه

توضیح: اجرای زیر رایگان است. سنجهٔ تصمیم در بازپخش فقط یک شاخص مسیر است؛ کیفیت مدل واقعی، درستی علت و مفیدبودن پاسخ را ثابت نمی‌کند. آزمون نهایی باید پس از قفل تنظیمات اجرا شود.

In [6]:
subprocess.run([sys.executable, 'scripts/run_tests.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/evaluate.py', '--split', 'dev'], cwd=ROOT, check=True)

CompletedProcess(args=['C:\\Users\\Rasa\\Documents\\ChatGPT\\Quera Agentic AI\\builds\\streamlit-casepilot\\.venv\\Scripts\\python.exe', 'scripts/evaluate.py', '--split', 'dev'], returncode=0)

## مرحلهٔ ارزیابی خصمانه

توضیح: ورودی‌های تزریقی و مجوزهای جعلی در چند خانواده آزموده می‌شوند. بازپخش مقاومت مدل واقعی در برابر تزریق را اثبات نمی‌کند؛ نتیجهٔ مکانیزم مجوز و استناد جدا گزارش می‌شود.

In [7]:
subprocess.run([sys.executable, 'scripts/adversarial.py'], cwd=ROOT, check=True)

CompletedProcess(args=['C:\\Users\\Rasa\\Documents\\ChatGPT\\Quera Agentic AI\\builds\\streamlit-casepilot\\.venv\\Scripts\\python.exe', 'scripts/adversarial.py'], returncode=0)

## مرحلهٔ اتصال واقعی پس از دریافت کلید

راهنما: کلید و نام مدل و تعرفه را فقط در محیط تنظیم کنید. ابتدا نمونهٔ کوچک توسعه اجرا شود. مقدار زیر پیش‌فرض خاموش است. راهنمای دقیق در فایل `docs/LIVE_RUN_FA.md` آمده است.

In [8]:
RUN_LIVE = False
if RUN_LIVE:
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--mode', 'live', '--split', 'dev', '--limit', '2', '--no-scenarios'], cwd=ROOT, check=True)
else:
    print('وضعیت: اجرای واقعی انجام نشده؛ کلید هنوز دریافت نشده است.')
temporary.cleanup()

وضعیت: اجرای واقعی انجام نشده؛ کلید هنوز دریافت نشده است.
